In [2]:
import pandas as pd
import sqlite3
import os

DB_PATH = r'..\data\db\fintech.db'
RAW_DIR = r'..\data\raw'

conn = sqlite3.connect(DB_PATH)

# 1. users, cards 먼저
print("1/3 users, cards 적재 중...")
users = pd.read_csv(os.path.join(RAW_DIR, 'sd254_users.csv'))
cards = pd.read_csv(os.path.join(RAW_DIR, 'sd254_cards.csv'))
users.to_sql('users', conn, if_exists='replace', index=False)
cards.to_sql('cards', conn, if_exists='replace', index=False)
print(f"users: {len(users):,}행 / cards: {len(cards):,}행 완료")

# 2. transactions chunk 스트리밍
print("2/3 transactions 적재 중... (10~20분 소요)")
chunk_size = 100000
total = 0
first_chunk = True

for chunk in pd.read_csv(
    os.path.join(RAW_DIR, 'credit_card_transactions-ibm_v2.csv'),
    chunksize=chunk_size
):
    chunk.to_sql(
        'transactions',
        conn,
        if_exists='replace' if first_chunk else 'append',
        index=False
    )
    total += len(chunk)
    first_chunk = False
    print(f"  적재 완료: {total:,}행", end='\r')

print(f"\n3/3 완료! 총 {total:,}행")

# 3. 확인
for table in ['transactions', 'users', 'cards']:
    count = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {table}", conn)
    print(f"{table}: {count['cnt'].values[0]:,}행")

conn.close()

1/3 users, cards 적재 중...
users: 2,000행 / cards: 6,146행 완료
2/3 transactions 적재 중... (10~20분 소요)
  적재 완료: 24,386,900행
3/3 완료! 총 24,386,900행
transactions: 24,386,900행
users: 2,000행
cards: 6,146행
